# Official Score: Full-Data XGBoost + Zindi Submission
**Zindi Urban Air Pollution Challenge — PM2.5 Prediction**

Internal validation (`02_model_xgboost.ipynb`) used an 80/20 location-based split to compare models and tune hyperparameters. For an **official leaderboard score**, that's no longer the goal — we want the best possible model on the real, unseen `Test.csv` locations.

Two changes from the internal-validation notebook:
1. **Train on all 340 Train.csv locations** (no held-out split) — more training data, better final model. The internal validation already told us how well this approach generalizes; we don't need to hold out data anymore.
2. **Reuse the best hyperparameters already found** via `RandomizedSearchCV` (`02_model_xgboost.ipynb`) instead of re-running the expensive search — they were already cross-validated on 5 folds, re-searching on full data wouldn't meaningfully change them and costs significant time.

**Important gap this notebook fills:** the EDA pipeline (`01_eda.ipynb`) was only ever applied to `Train.csv`. `Test.csv` needs the *exact same* feature engineering — same column drops, same zero-to-NaN corrections, same flags — or the model will see a different feature space at prediction time than it was trained on. The functions below replicate that pipeline so both datasets go through identical steps.

In [31]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

RANDOM_STATE = 42

pd.set_option('display.max_columns', None)

In [32]:
df_train = pd.read_csv("data/Train.csv")
df_test = pd.read_csv("data/Test.csv")

print("Train:", df_train.shape, "| Test:", df_test.shape)

Train: (30557, 82) | Test: (16136, 77)


## Shared Feature Engineering Pipeline

Same parameters as `01_eda.ipynb` / `02_model_xgboost.ipynb`, now defined once and applied to both `df_train` and `df_test` via functions — this is what guarantees train/test consistency.

In [33]:
# Same constants as 01_eda.ipynb
BASE_COLS = ["Place_ID", "Date", "Place_ID X Date"]

DROP_REDUNDANT = [
    "L3_CLOUD_cloud_base_height",
    "L3_CLOUD_cloud_base_pressure",
    "L3_CLOUD_cloud_top_pressure",
    "L3_NO2_NO2_column_number_density",
    "L3_NO2_NO2_slant_column_number_density",
    "L3_HCHO_HCHO_slant_column_number_density",
]

# From 02_model_xgboost.ipynb - excluded due to the sensor_altitude / orbital-proxy finding
DROP_METADATA = ["L3_AER_AI_sensor_altitude", "L3_NO2_sensor_altitude", "L3_CO_sensor_altitude"]

WEATHER_COLS = [
    "precipitable_water_entire_atmosphere",
    "relative_humidity_2m_above_ground",
    "specific_humidity_2m_above_ground",
    "temperature_2m_above_ground",
    "u_component_of_wind_10m_above_ground",
    "v_component_of_wind_10m_above_ground",
]

# Built once from df_train's columns - df_test has the same predictor columns (no target)
_angle_cols = [c for c in df_train.columns if c.endswith("_angle")]
_sat_cols_all = [c for c in df_train.columns if c.startswith("L3_")]
_sat_cols = [c for c in _sat_cols_all if c not in _angle_cols]

FEATURE_GROUPS = {
    "NO2": [c for c in _sat_cols if c.startswith("L3_NO2_")],
    "O3": [c for c in _sat_cols if c.startswith("L3_O3_")],
    "CO": [c for c in _sat_cols if c.startswith("L3_CO_")],
    "HCHO": [c for c in _sat_cols if c.startswith("L3_HCHO_")],
    "CLOUD": [c for c in _sat_cols if c.startswith("L3_CLOUD_")],
    "AER_AI": [c for c in _sat_cols if c.startswith("L3_AER_AI_")],
    "SO2": [c for c in _sat_cols if c.startswith("L3_SO2_")],
    "CH4": [c for c in _sat_cols if c.startswith("L3_CH4_")],
}

In [34]:
def build_satellite_features(df):
    """Replicates the satellite EDA pipeline from 01_eda.ipynb. Works for Train (has target)
    and Test (no target) alike."""
    has_target = "target" in df.columns
    keep_cols = BASE_COLS + _sat_cols_all + (["target"] if has_target else [])
    df_sat = df[keep_cols].copy()
    df_sat = df_sat.drop(columns=_angle_cols)

    # NO2: zero -> NaN, then flags
    no2_cols = FEATURE_GROUPS["NO2"]
    for col in no2_cols:
        df_sat[col] = df_sat[col].replace(0, np.nan)
    df_sat["no2_measured"] = df_sat[no2_cols[0]].notna().astype(int)
    df_sat["no2_tropospheric_derived"] = df_sat["L3_NO2_tropospheric_NO2_column_number_density"].notna().astype(int)

    # CH4: zero -> NaN, then flag
    ch4_cols = FEATURE_GROUPS["CH4"]
    for col in ch4_cols:
        df_sat[col] = df_sat[col].replace(0, np.nan)
    df_sat["ch4_measured"] = df_sat[ch4_cols[0]].notna().astype(int)

    # CO, HCHO, SO2: flags only (no zero-fix needed - see slide "One pattern, three more pollutants")
    for group, flag_name in [("CO", "co_measured"), ("HCHO", "hcho_measured"), ("SO2", "so2_measured")]:
        cols = FEATURE_GROUPS[group]
        df_sat[flag_name] = df_sat[cols[0]].notna().astype(int)

    # df_sat = df_sat.drop(columns=DROP_REDUNDANT + DROP_METADATA, errors="ignore")
    df_sat = df_sat.drop(columns=DROP_REDUNDANT, errors="ignore")
    return df_sat


def build_weather_features(df):
    """Replicates the weather EDA pipeline from 01_eda.ipynb."""
    df_weather = df[BASE_COLS + WEATHER_COLS].copy()
    df_weather = df_weather.drop(columns="specific_humidity_2m_above_ground")

    u = df_weather["u_component_of_wind_10m_above_ground"]
    v = df_weather["v_component_of_wind_10m_above_ground"]
    df_weather["wind_speed"] = np.sqrt(u**2 + v**2)
    wind_direction = (np.degrees(np.arctan2(v, u)) + 360) % 360
    rad = np.radians(wind_direction)
    df_weather["wind_dir_sin"] = np.sin(rad)
    df_weather["wind_dir_cos"] = np.cos(rad)
    return df_weather

In [35]:
df_sat_train = build_satellite_features(df_train)
df_weather_train = build_weather_features(df_train)

df_sat_test = build_satellite_features(df_test)
df_weather_test = build_weather_features(df_test)

print("df_sat_train:", df_sat_train.shape, "| df_sat_test:", df_sat_test.shape)
print("df_weather_train:", df_weather_train.shape, "| df_weather_test:", df_weather_test.shape)

df_sat_train: (30557, 40) | df_sat_test: (16136, 39)
df_weather_train: (30557, 11) | df_weather_test: (16136, 11)


In [36]:
df_weather_train_features = df_weather_train.drop(columns=["target"], errors="ignore")
df_combined_train = df_sat_train.merge(
    df_weather_train_features, on=["Place_ID", "Date", "Place_ID X Date"], how="inner", validate="1:1"
)

df_combined_test = df_sat_test.merge(
    df_weather_test, on=["Place_ID", "Date", "Place_ID X Date"], how="inner", validate="1:1"
)

assert len(df_combined_train) == len(df_sat_train) == len(df_weather_train), "Rows lost during train merge!"
assert len(df_combined_test) == len(df_sat_test) == len(df_weather_test), "Rows lost during test merge!"

print("df_combined_train:", df_combined_train.shape)
print("df_combined_test: ", df_combined_test.shape)

df_combined_train: (30557, 48)
df_combined_test:  (16136, 47)


### Consistency Check (critical)

Train and Test must end up with **exactly the same feature columns** (Train has `target` extra, Test doesn't) — otherwise the model trained on Train won't even be able to predict on Test, or worse, will silently misalign columns.

In [37]:
train_feature_cols = set(df_combined_train.columns) - {"target", "Place_ID", "Date", "Place_ID X Date"}
test_feature_cols = set(df_combined_test.columns) - {"Place_ID", "Date", "Place_ID X Date"}

assert train_feature_cols == test_feature_cols, (
    f"Column mismatch!\nIn train only: {train_feature_cols - test_feature_cols}\n"
    f"In test only: {test_feature_cols - train_feature_cols}"
)
print(f"OK - {len(train_feature_cols)} feature columns match exactly between train and test.")

OK - 44 feature columns match exactly between train and test.


## Train Final Model on Full Train Data

In [38]:
drop_cols = ["target", "Place_ID", "Date", "Place_ID X Date"]

X_full_train = df_combined_train.drop(columns=drop_cols, errors="ignore")
y_full_train = df_combined_train["target"]

X_submission = df_combined_test.drop(columns=["Place_ID", "Date", "Place_ID X Date"])

# Column order must match exactly for XGBoost
X_submission = X_submission[X_full_train.columns]

print("X_full_train:", X_full_train.shape, "| X_submission:", X_submission.shape)

X_full_train: (30557, 44) | X_submission: (16136, 44)


In [39]:
# Best hyperparameters found via RandomizedSearchCV in 02_model_xgboost.ipynb
# (GroupKFold(5), n_iter=100) - reused here rather than re-running the search on full data
best_params = {
    'colsample_bytree': 0.7390476857930735,
    'learning_rate': 0.019223357630697886,
    'max_depth': 8,
    'min_child_weight': 1,
    'n_estimators': 744,
    'reg_lambda': 2.573504456147266,
    'subsample': 0.682533487362317,
}

final_model = XGBRegressor(random_state=RANDOM_STATE, **best_params)
final_model.fit(X_full_train, y_full_train)
print("Final model trained on all", len(X_full_train), "rows.")

Final model trained on all 30557 rows.


## Predict on Test.csv and Build Submission File

In [ ]:
predictions = final_model.predict(X_submission)
predictions = np.clip(predictions, 0, None)  # PM2.5 can't be negative

submission = pd.DataFrame({
    "Place_ID X Date": df_test["Place_ID X Date"],
    "target": predictions,
})

# Sanity checks before exporting
assert len(submission) == len(df_test), "Row count mismatch with Test.csv!"
assert submission["target"].isna().sum() == 0, "NaN predictions present!"
assert (submission["target"] >= 0).all(), "Negative predictions present!"

submission.to_csv("data/submission.csv", index=False)
print(submission.shape)
submission.head()

(16136, 2)


,Place_ID X Date,target
0,0OS9LVX X 2020-01-02,41.852161
1,0OS9LVX X 2020-01-03,31.958303
2,0OS9LVX X 2020-01-04,30.383558
3,0OS9LVX X 2020-01-05,34.759148
4,0OS9LVX X 2020-01-06,42.670338


: 

## Submitting to Zindi

1. Open the challenge page on [zindi.africa](https://zindi.africa) and go to the **Submit** tab.
2. Upload `data/submission.csv` — column names and row order should match `SampleSubmission.csv` (`Place_ID X Date`, `target`). Zindi typically matches by the ID column, not strict row order, but matching anyway avoids surprises.
3. Check the **Submission Limits** on the rules page first — most Zindi challenges cap submissions per day (commonly 5-10, varies by competition). Don't burn an attempt before you're confident in the file.
4. After upload, Zindi auto-scores against the hidden Test.csv targets using the competition's metric (RMSE here) and posts the **public leaderboard** score.
5. If you submit more than once, use **"Make Submission Active"** (or equivalent) to mark which submission counts for final ranking — Zindi doesn't always default to your best or most recent one.
6. Note: the public leaderboard is often computed on only part of the test set, with a private leaderboard (final ranking) revealed after the competition closes — don't over-index on small public-score differences.